# Data Preparation Protocol

## Purpose

This notebook defines and validates the data-preparation procedure used
before development sampling and model evaluation.

It separates data preparation from random sampling so that eligibility,
duplicate handling, feature inclusion and leakage prevention are decided
before any records are selected.

## Guiding principles

1. The provider-supplied raw files remain unchanged.
2. Every transformation must be explicit and reproducible.
3. Records must not be removed without documenting the reason.
4. `Label` and `Attack` are ground-truth fields and must never appear in
   model inputs.
5. Structured-value and deterministic-text conditions must contain the
   same underlying features, values, units and numerical precision.
6. Sampling occurs only after the eligible population has been defined.
7. Data preparation for Logistic Regression and Random Forest must avoid
   fitting preprocessing operations on evaluation data.

## Provisional scope

The initial development task is provisionally restricted to:

- `Benign`
- `DoS`

This same-source comparison is a development pilot. The final dataset
design remains subject to clarification of the assignment's
measured-benign-trace requirement.

In [1]:
# ---------------------------------------------------------------------
# Import the tools required for schema and data-preparation analysis
# ---------------------------------------------------------------------

from pathlib import Path
import platform
import sys

import numpy as np
import pandas as pd


# ---------------------------------------------------------------------
# Locate the project and provider-supplied files
# ---------------------------------------------------------------------

# This notebook is stored inside notebooks/, so its parent directory is
# the project root.
PROJECT_ROOT = Path.cwd().resolve().parent

RAW_DATA_DIR = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "nf_unsw_nb15_v3"
)

SOURCE_CSV = RAW_DATA_DIR / "NF-UNSW-NB15-v3.csv"
FEATURE_DEFINITIONS_CSV = (
    RAW_DATA_DIR / "NetFlow_v3_Features.csv"
)

assert SOURCE_CSV.exists(), (
    f"Main dataset not found: {SOURCE_CSV}"
)

assert FEATURE_DEFINITIONS_CSV.exists(), (
    f"Feature-definition file not found: "
    f"{FEATURE_DEFINITIONS_CSV}"
)

path_check = pd.Series(
    {
        "project_root": str(PROJECT_ROOT),
        "source_csv_exists": SOURCE_CSV.exists(),
        "feature_definitions_exist": (
            FEATURE_DEFINITIONS_CSV.exists()
        ),
        "python_version": sys.version.split()[0],
        "pandas_version": pd.__version__,
        "numpy_version": np.__version__,
        "platform": platform.platform(),
    },
    name="value",
)

path_check

project_root                 /Users/ruiwang/Developer/compsci742-rui-pilot
source_csv_exists                                                     True
feature_definitions_exist                                             True
python_version                                                     3.11.14
pandas_version                                                       3.0.5
numpy_version                                                        2.4.6
platform                                      macOS-15.7.3-arm64-arm-64bit
Name: value, dtype: object

In [2]:
# ---------------------------------------------------------------------
# Inspect the provider-supplied feature-definition table
# ---------------------------------------------------------------------

# This small file documents the meaning of the NetFlow-v3 variables.
# We inspect it before deciding which columns can enter the experiment.
feature_definitions = pd.read_csv(
    FEATURE_DEFINITIONS_CSV
)

print(
    "Feature-definition dimensions: "
    f"{feature_definitions.shape[0]} rows × "
    f"{feature_definitions.shape[1]} columns"
)

print("\nDefinition-table column names:")
print(feature_definitions.columns.tolist())

# Show the complete table rather than only its first five entries.
with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    120,
):
    display(feature_definitions)

Feature-definition dimensions: 53 rows × 2 columns

Definition-table column names:
['Feature', 'Description']


,Feature,Description
0,IPV4_SRC_ADDR,IPv4 source address
1,IPV4_DST_ADDR,IPv4 destination address
2,L4_SRC_PORT,IPv4 source port number
3,L4_DST_PORT,IPv4 destination port number
4,PROTOCOL,IP protocol identifier byte
5,L7_PROTO,Layer 7 protocol (numeric)
6,IN_BYTES,Incoming number of bytes
7,OUT_BYTES,Outgoing number of bytes
8,IN_PKTS,Incoming number of packets
9,OUT_PKTS,Outgoing number of packets


## Schema alignment and documentation audit

The main dataset contains 55 columns, whereas the provider-supplied
feature-definition table contains 53 feature entries. The expected
difference is the two ground-truth columns, `Label` and `Attack`.

Before assigning feature roles, the following checks compare the source
CSV schema with the provider documentation and identify undocumented,
unused or inconsistently described fields.

In [3]:
# ---------------------------------------------------------------------
# Compare the main CSV schema with the provider's feature definitions
# ---------------------------------------------------------------------

# Read only the CSV header. No network-flow records are loaded, so this
# operation is fast and uses negligible memory.
source_columns = pd.read_csv(
    SOURCE_CSV,
    nrows=0,
).columns.tolist()

# The provider table is expected to use its first column for feature
# names and its second column for descriptions.
definition_name_column = feature_definitions.columns[0]
definition_description_column = feature_definitions.columns[1]

documented_features = (
    feature_definitions[definition_name_column]
    .astype(str)
    .str.strip()
    .tolist()
)

# Ground-truth fields are expected in the main CSV but should not appear
# among the predictor definitions.
GROUND_TRUTH_COLUMNS = ["Label", "Attack"]

source_predictor_columns = [
    column
    for column in source_columns
    if column not in GROUND_TRUTH_COLUMNS
]

# Find discrepancies in both directions.
undocumented_source_columns = [
    column
    for column in source_predictor_columns
    if column not in documented_features
]

documented_but_absent_columns = [
    column
    for column in documented_features
    if column not in source_predictor_columns
]

schema_alignment_summary = pd.Series(
    {
        "source_csv_columns": len(source_columns),
        "source_predictor_columns": len(
            source_predictor_columns
        ),
        "ground_truth_columns": len(
            GROUND_TRUTH_COLUMNS
        ),
        "documented_features": len(
            documented_features
        ),
        "undocumented_source_predictors": len(
            undocumented_source_columns
        ),
        "documented_but_absent": len(
            documented_but_absent_columns
        ),
        "predictor_sets_match": (
            set(source_predictor_columns)
            == set(documented_features)
        ),
        "predictor_order_matches": (
            source_predictor_columns
            == documented_features
        ),
    },
    name="value",
)

schema_alignment_summary

source_csv_columns                   55
source_predictor_columns             53
ground_truth_columns                  2
documented_features                  53
undocumented_source_predictors        0
documented_but_absent                 0
predictor_sets_match               True
predictor_order_matches           False
Name: value, dtype: object

In [4]:
# ---------------------------------------------------------------------
# Display any schema discrepancies explicitly
# ---------------------------------------------------------------------

schema_discrepancies = pd.DataFrame(
    {
        "discrepancy_type": (
            ["present_in_csv_but_not_documented"]
            * len(undocumented_source_columns)
            +
            ["documented_but_absent_from_csv"]
            * len(documented_but_absent_columns)
        ),
        "column_name": (
            undocumented_source_columns
            + documented_but_absent_columns
        ),
    }
)

if schema_discrepancies.empty:
    print(
        "No predictor-name discrepancies were detected. "
        "All 53 source predictor columns have provider definitions."
    )
else:
    display(schema_discrepancies)

No predictor-name discrepancies were detected. All 53 source predictor columns have provider definitions.


In [6]:
# ---------------------------------------------------------------------
# Audit provider descriptions for encoding and statistic-name problems
# ---------------------------------------------------------------------

definitions_review = feature_definitions[
    [
        definition_name_column,
        definition_description_column,
    ]
].copy()

definitions_review.columns = [
    "column_name",
    "provider_description",
]

# Normalise surrounding whitespace before performing text comparisons.
# This prevents invisible spaces in the provider file from causing a
# genuine documentation problem to be missed.
definitions_review["column_name"] = (
    definitions_review["column_name"]
    .astype(str)
    .str.strip()
)

definitions_review["provider_description"] = (
    definitions_review["provider_description"]
    .fillna("")
    .astype(str)
    .str.strip()
)

description_text = definitions_review[
    "provider_description"
]

# Detect visible Unicode replacement characters caused by broken text
# encoding in the provider-supplied feature-definition file.
definitions_review["possible_encoding_problem"] = (
    description_text.str.contains(
        "\ufffd|��",
        regex=True,
    )
)

# Expected statistic wording for the inter-packet-arrival-time fields.
expected_iat_terms = {
    "DST_TO_SRC_IAT_MIN": "minimum",
    "DST_TO_SRC_IAT_MAX": "maximum",
    "DST_TO_SRC_IAT_AVG": "average",
    "DST_TO_SRC_IAT_STDDEV": "standard deviation",
}

definitions_review["expected_iat_term"] = (
    definitions_review["column_name"]
    .map(expected_iat_terms)
)

# A row is suspicious when it is one of the four destination-to-source
# IAT statistics but its description does not contain the statistic
# implied by the feature name.
definitions_review["possible_iat_description_problem"] = (
    definitions_review["expected_iat_term"].notna()
    & ~definitions_review.apply(
        lambda row: (
            row["expected_iat_term"]
            in row["provider_description"].lower()
        )
        if pd.notna(row["expected_iat_term"])
        else True,
        axis=1,
    )
)

definitions_requiring_review = definitions_review[
    definitions_review[
        [
            "possible_encoding_problem",
            "possible_iat_description_problem",
        ]
    ].any(axis=1)
].copy()

definitions_requiring_review

,column_name,provider_description,possible_encoding_problem,expected_iat_term,possible_iat_description_problem
34,NUM_PKTS_1024_TO_1514_BYTES,Packets whose IP size >��1024 and <= 1514,True,NaN,False
50,DST_TO_SRC_IAT_MAX,Minimum Inter-Packet Arrval Time (dst > src),False,maximum,True
51,DST_TO_SRC_IAT_AVG,Minimum Inter-Packet Arrval Time (dst > src),False,average,True
52,DST_TO_SRC_IAT_STDDEV,Minimum Inter-Packet Arrval Time (dst > src),False,standard deviation,True


## Provisional feature roles

Not every source column should be presented to a model.

The following table separates:

- ground-truth fields used only for evaluation;
- direct identifiers excluded from the core model input;
- absolute timestamps excluded to reduce temporal and dataset-specific
  shortcut learning;
- transaction identifiers excluded because their numeric values do not
  represent ordered network behaviour;
- encoded categorical fields requiring semantic handling; and
- quantitative candidate features.

These assignments are provisional and will be validated before the
feature set is frozen.

In [7]:
# ---------------------------------------------------------------------
# Define provisional experimental roles for all 55 source columns
# ---------------------------------------------------------------------

# These columns contain the answer and must never appear in model input.
ground_truth_columns = [
    "Label",
    "Attack",
]

# Raw IP addresses directly identify endpoints and may allow the model
# to memorise dataset-specific address patterns rather than reason from
# network-flow behaviour.
direct_identifier_columns = [
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
]

# Absolute timestamps are useful for provenance and temporal splitting,
# but are initially excluded from per-record model input. Flow duration
# remains available as a behavioural feature.
absolute_timestamp_columns = [
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
]

# A DNS transaction ID identifies a request-response transaction but its
# numeric magnitude does not describe anomalous network behaviour.
transaction_identifier_columns = [
    "DNS_QUERY_ID",
]

# These fields contain numeric codes representing categories, protocols,
# flags or return values. They should not automatically be interpreted
# as continuous quantities by Logistic Regression.
encoded_categorical_columns = [
    "L4_SRC_PORT",
    "L4_DST_PORT",
    "PROTOCOL",
    "L7_PROTO",
    "TCP_FLAGS",
    "CLIENT_TCP_FLAGS",
    "SERVER_TCP_FLAGS",
    "ICMP_TYPE",
    "ICMP_IPV4_TYPE",
    "DNS_QUERY_TYPE",
    "FTP_COMMAND_RET_CODE",
]


def assign_feature_role(column_name):
    """
    Assign a provisional experimental role to one source column.

    This function records decisions without modifying the raw dataset.
    """
    if column_name in ground_truth_columns:
        return "ground_truth"

    if column_name in direct_identifier_columns:
        return "direct_identifier"

    if column_name in absolute_timestamp_columns:
        return "temporal_provenance"

    if column_name in transaction_identifier_columns:
        return "transaction_identifier"

    if column_name in encoded_categorical_columns:
        return "encoded_categorical_candidate"

    return "quantitative_candidate"


def assign_core_decision(column_name):
    """
    Record whether a column is included in the provisional core input.
    """
    if column_name in ground_truth_columns:
        return "exclude_from_model"

    if column_name in direct_identifier_columns:
        return "exclude_from_core"

    if column_name in absolute_timestamp_columns:
        return "exclude_from_core"

    if column_name in transaction_identifier_columns:
        return "exclude_from_core"

    if column_name in encoded_categorical_columns:
        return "requires_encoding_review"

    return "provisional_include"


def explain_feature_decision(column_name):
    """
    Provide a readable reason for the provisional role assignment.
    """
    if column_name in ground_truth_columns:
        return (
            "Ground truth used only for evaluation; including it would "
            "cause direct label leakage."
        )

    if column_name in direct_identifier_columns:
        return (
            "Direct endpoint identifier with high dataset-specific "
            "shortcut-learning risk."
        )

    if column_name in absolute_timestamp_columns:
        return (
            "Retained for provenance and possible temporal splitting, "
            "but excluded from per-record core input."
        )

    if column_name in transaction_identifier_columns:
        return (
            "Transaction identifier whose numeric magnitude is not an "
            "ordered behavioural measurement."
        )

    if column_name in encoded_categorical_columns:
        return (
            "Numeric code or flag requiring semantic treatment rather "
            "than automatic continuous interpretation."
        )

    return (
        "Quantitative network-flow measurement provisionally eligible "
        "for the core feature set."
    )


# Convert the provider definitions to a column-name-to-description map.
provider_description_map = dict(
    zip(
        definitions_review["column_name"],
        definitions_review["provider_description"],
    )
)

feature_roles = pd.DataFrame(
    {
        "column_position": range(1, len(source_columns) + 1),
        "column_name": source_columns,
    }
)

feature_roles["provider_description"] = (
    feature_roles["column_name"]
    .map(provider_description_map)
)

# Label and Attack are not included in the 53-row provider dictionary,
# so give them explicit descriptions.
feature_roles.loc[
    feature_roles["column_name"] == "Label",
    "provider_description",
] = "Binary ground-truth label"

feature_roles.loc[
    feature_roles["column_name"] == "Attack",
    "provider_description",
] = "Ground-truth traffic or attack category"

feature_roles["feature_role"] = (
    feature_roles["column_name"]
    .map(assign_feature_role)
)

feature_roles["core_decision"] = (
    feature_roles["column_name"]
    .map(assign_core_decision)
)

feature_roles["decision_reason"] = (
    feature_roles["column_name"]
    .map(explain_feature_decision)
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    90,
):
    display(feature_roles)

,column_position,column_name,provider_description,feature_role,core_decision,decision_reason
0,1,FLOW_START_MILLISECONDS,Flow start timestamp in milliseconds,temporal_provenance,exclude_from_core,"Retained for provenance and possible temporal splitting, but excluded from per-record ..."
1,2,FLOW_END_MILLISECONDS,Flow end timestamp in milliseconds,temporal_provenance,exclude_from_core,"Retained for provenance and possible temporal splitting, but excluded from per-record ..."
2,3,IPV4_SRC_ADDR,IPv4 source address,direct_identifier,exclude_from_core,Direct endpoint identifier with high dataset-specific shortcut-learning risk.
3,4,L4_SRC_PORT,IPv4 source port number,encoded_categorical_candidate,requires_encoding_review,Numeric code or flag requiring semantic treatment rather than automatic continuous int...
4,5,IPV4_DST_ADDR,IPv4 destination address,direct_identifier,exclude_from_core,Direct endpoint identifier with high dataset-specific shortcut-learning risk.
5,6,L4_DST_PORT,IPv4 destination port number,encoded_categorical_candidate,requires_encoding_review,Numeric code or flag requiring semantic treatment rather than automatic continuous int...
6,7,PROTOCOL,IP protocol identifier byte,encoded_categorical_candidate,requires_encoding_review,Numeric code or flag requiring semantic treatment rather than automatic continuous int...
7,8,L7_PROTO,Layer 7 protocol (numeric),encoded_categorical_candidate,requires_encoding_review,Numeric code or flag requiring semantic treatment rather than automatic continuous int...
8,9,IN_BYTES,Incoming number of bytes,quantitative_candidate,provisional_include,Quantitative network-flow measurement provisionally eligible for the core feature set.
9,10,IN_PKTS,Incoming number of packets,quantitative_candidate,provisional_include,Quantitative network-flow measurement provisionally eligible for the core feature set.


In [8]:
# ---------------------------------------------------------------------
# Summarise the provisional feature-role assignments
# ---------------------------------------------------------------------

feature_role_summary = (
    feature_roles
    .groupby(
        ["feature_role", "core_decision"],
        dropna=False,
    )
    .size()
    .rename("column_count")
    .reset_index()
)

feature_role_summary

,feature_role,core_decision,column_count
0,direct_identifier,exclude_from_core,2
1,encoded_categorical_candidate,requires_encoding_review,11
2,ground_truth,exclude_from_model,2
3,quantitative_candidate,provisional_include,37
4,temporal_provenance,exclude_from_core,2
5,transaction_identifier,exclude_from_core,1


In [9]:
assert len(feature_roles) == 55
assert feature_roles["column_name"].is_unique
assert feature_roles["feature_role"].notna().all()
assert feature_roles["core_decision"].notna().all()

print(
    "All 55 source columns received exactly one provisional role."
)

All 55 source columns received exactly one provisional role.


## Encoded-field population audit

Several source fields are stored numerically but represent protocol
codes, port identifiers, flag combinations, query types or return codes.

Treating these fields as ordinary continuous variables could create
invalid numeric relationships. The following audit therefore examines:

- the number of distinct observed values;
- missing-value counts;
- the most frequent values within Benign and DoS records; and
- whether any encoded field has very high cardinality.

Only the provisional Benign and DoS populations are scanned at this
stage. No data values are modified.

In [10]:
# ---------------------------------------------------------------------
# Audit encoded categorical fields in the provisional Benign/DoS scope
# ---------------------------------------------------------------------

from collections import Counter


TARGET_CATEGORIES = ["Benign", "DoS"]
CHUNK_SIZE = 100_000

# Only the category column and the 11 encoded candidate fields are read.
# This is faster and more memory-efficient than loading all 55 columns.
encoded_audit_columns = [
    "Attack",
    *encoded_categorical_columns,
]

# One frequency counter is maintained for every category/feature pair.
# A Counter allows frequencies from successive chunks to be added.
encoded_value_counters = {
    category: {
        column: Counter()
        for column in encoded_categorical_columns
    }
    for category in TARGET_CATEGORIES
}

category_rows_scanned = Counter()
rows_processed = 0


for chunk_number, chunk in enumerate(
    pd.read_csv(
        SOURCE_CSV,
        usecols=encoded_audit_columns,
        chunksize=CHUNK_SIZE,
    ),
    start=1,
):
    # Restrict this audit to the provisional binary task.
    target_chunk = chunk[
        chunk["Attack"].isin(TARGET_CATEGORIES)
    ]

    for category in TARGET_CATEGORIES:
        category_chunk = target_chunk[
            target_chunk["Attack"] == category
        ]

        category_rows_scanned[category] += len(
            category_chunk
        )

        for column in encoded_categorical_columns:
            # Convert values to strings only for frequency reporting.
            # The original source data remains unchanged.
            #
            # A fixed token is used for missing values because NaN values
            # are awkward dictionary keys and may not compare reliably.
            report_values = (
                category_chunk[column]
                .astype("string")
                .fillna("<MISSING>")
            )

            value_counts = (
                report_values
                .value_counts(dropna=False)
                .to_dict()
            )

            encoded_value_counters[
                category
            ][column].update(value_counts)

    rows_processed += len(chunk)

    if chunk_number == 1 or chunk_number % 5 == 0:
        print(
            f"Processed chunk {chunk_number:>2}: "
            f"{rows_processed:,} cumulative rows"
        )


print("\nEncoded-field audit completed.")
print(f"Total source rows scanned: {rows_processed:,}")
print(
    "Benign rows examined: "
    f"{category_rows_scanned['Benign']:,}"
)
print(
    "DoS rows examined: "
    f"{category_rows_scanned['DoS']:,}"
)

Processed chunk  1: 100,000 cumulative rows
Processed chunk  5: 500,000 cumulative rows
Processed chunk 10: 1,000,000 cumulative rows
Processed chunk 15: 1,500,000 cumulative rows
Processed chunk 20: 2,000,000 cumulative rows

Encoded-field audit completed.
Total source rows scanned: 2,365,424
Benign rows examined: 2,237,731
DoS rows examined: 5,980


In [11]:
# ---------------------------------------------------------------------
# Summarise cardinality and missing values by field and category
# ---------------------------------------------------------------------

encoded_cardinality_rows = []

for category in TARGET_CATEGORIES:
    category_total = category_rows_scanned[category]

    for column in encoded_categorical_columns:
        counter = encoded_value_counters[
            category
        ][column]

        missing_count = counter.get(
            "<MISSING>",
            0,
        )

        # The missing token is not counted as an observed category value.
        observed_unique_values = sum(
            value != "<MISSING>"
            for value in counter
        )

        encoded_cardinality_rows.append(
            {
                "Attack": category,
                "column_name": column,
                "unique_non_missing_values": (
                    observed_unique_values
                ),
                "missing_count": missing_count,
                "missing_percentage": (
                    100 * missing_count / category_total
                    if category_total
                    else np.nan
                ),
            }
        )

encoded_cardinality_summary = pd.DataFrame(
    encoded_cardinality_rows
)

encoded_cardinality_summary

,Attack,column_name,unique_non_missing_values,missing_count,missing_percentage
0,Benign,L4_SRC_PORT,64530,0,0.0
1,Benign,L4_DST_PORT,64516,0,0.0
2,Benign,PROTOCOL,4,0,0.0
3,Benign,L7_PROTO,91,0,0.0
4,Benign,TCP_FLAGS,13,0,0.0
5,Benign,CLIENT_TCP_FLAGS,12,0,0.0
6,Benign,SERVER_TCP_FLAGS,12,0,0.0
7,Benign,ICMP_TYPE,521,0,0.0
8,Benign,ICMP_IPV4_TYPE,256,0,0.0
9,Benign,DNS_QUERY_TYPE,11,0,0.0


In [12]:
# ---------------------------------------------------------------------
# Show the five most frequent values per category and encoded field
# ---------------------------------------------------------------------

TOP_VALUES_TO_SHOW = 5
top_value_rows = []

for category in TARGET_CATEGORIES:
    category_total = category_rows_scanned[category]

    for column in encoded_categorical_columns:
        most_common_values = encoded_value_counters[
            category
        ][column].most_common(TOP_VALUES_TO_SHOW)

        for rank, (value, count) in enumerate(
            most_common_values,
            start=1,
        ):
            top_value_rows.append(
                {
                    "Attack": category,
                    "column_name": column,
                    "rank": rank,
                    "value": value,
                    "count": count,
                    "percentage_within_category": (
                        100 * count / category_total
                        if category_total
                        else np.nan
                    ),
                }
            )

encoded_top_values = pd.DataFrame(
    top_value_rows
)

with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
):
    display(encoded_top_values)

,Attack,column_name,rank,value,count,percentage_within_category
0,Benign,L4_SRC_PORT,1,1043,13170,0.588543
1,Benign,L4_SRC_PORT,2,47439,13123,0.586442
2,Benign,L4_SRC_PORT,3,0,1560,0.069713
3,Benign,L4_SRC_PORT,4,21,918,0.041024
4,Benign,L4_SRC_PORT,5,53,525,0.023461
5,Benign,L4_DST_PORT,1,21,470213,21.012937
6,Benign,L4_DST_PORT,2,53,366530,16.379538
7,Benign,L4_DST_PORT,3,80,179164,8.006503
8,Benign,L4_DST_PORT,4,6881,115744,5.172382
9,Benign,L4_DST_PORT,5,5190,110030,4.917034


## Provisional core-input policy

The provisional core model input contains 48 of the 55 source columns.

Seven columns are excluded:

- `Label` and `Attack`, because they contain the ground truth;
- the source and destination IPv4 addresses, because they are direct
  identifiers with dataset-specific shortcut-learning risk;
- the absolute flow start and end timestamps, because the core task
  concerns per-flow behaviour rather than collection date; and
- `DNS_QUERY_ID`, because it is a transaction identifier rather than an
  ordered behavioural measurement.

The remaining 48 columns consist of 37 quantitative measurements and
11 encoded categorical candidates.

Records containing missing or infinite directional second-byte values
will not automatically be deleted. The audit showed that these values
are class-dependent, so complete-case deletion could introduce
selection bias.

For the LLM conditions, missing and infinite states will be represented
explicitly and equivalently in both structured and deterministic-text
inputs. Conventional-model preprocessing will be fitted using training
data only and documented separately.

In [13]:
# ---------------------------------------------------------------------
# Record the provisional core-input decision for every source column
# ---------------------------------------------------------------------

# Seven columns are retained for provenance or evaluation but excluded
# from the model-facing feature set.
excluded_core_columns = [
    "Label",
    "Attack",
    "IPV4_SRC_ADDR",
    "IPV4_DST_ADDR",
    "FLOW_START_MILLISECONDS",
    "FLOW_END_MILLISECONDS",
    "DNS_QUERY_ID",
]

# Every other source column is provisionally retained.
provisional_core_features = [
    column
    for column in source_columns
    if column not in excluded_core_columns
]

# Add the model-facing decision to the existing role table.
feature_roles["provisional_core_input"] = (
    feature_roles["column_name"].isin(
        provisional_core_features
    )
)


def assign_value_treatment(column_name):
    """
    Describe how each source field will be treated during preparation.

    This records the protocol only; it does not transform any values.
    """
    if column_name in ground_truth_columns:
        return "retain_as_ground_truth_only"

    if column_name in direct_identifier_columns:
        return "retain_for_audit_but_exclude_from_model"

    if column_name in absolute_timestamp_columns:
        return "retain_for_provenance_but_exclude_from_model"

    if column_name in transaction_identifier_columns:
        return "retain_for_audit_but_exclude_from_model"

    if column_name in encoded_categorical_columns:
        return "retain_raw_code_and_review_semantic_encoding"

    if column_name in [
        "SRC_TO_DST_SECOND_BYTES",
        "DST_TO_SRC_SECOND_BYTES",
    ]:
        return "retain_record_and_encode_nonfinite_state_explicitly"

    return "retain_quantitative_value"


feature_roles["value_treatment"] = (
    feature_roles["column_name"]
    .map(assign_value_treatment)
)

# Display the complete decision table.
with pd.option_context(
    "display.max_rows",
    None,
    "display.max_columns",
    None,
    "display.max_colwidth",
    90,
):
    display(
        feature_roles[
            [
                "column_position",
                "column_name",
                "feature_role",
                "provisional_core_input",
                "value_treatment",
            ]
        ]
    )

,column_position,column_name,feature_role,provisional_core_input,value_treatment
0,1,FLOW_START_MILLISECONDS,temporal_provenance,False,retain_for_provenance_but_exclude_from_model
1,2,FLOW_END_MILLISECONDS,temporal_provenance,False,retain_for_provenance_but_exclude_from_model
2,3,IPV4_SRC_ADDR,direct_identifier,False,retain_for_audit_but_exclude_from_model
3,4,L4_SRC_PORT,encoded_categorical_candidate,True,retain_raw_code_and_review_semantic_encoding
4,5,IPV4_DST_ADDR,direct_identifier,False,retain_for_audit_but_exclude_from_model
5,6,L4_DST_PORT,encoded_categorical_candidate,True,retain_raw_code_and_review_semantic_encoding
6,7,PROTOCOL,encoded_categorical_candidate,True,retain_raw_code_and_review_semantic_encoding
7,8,L7_PROTO,encoded_categorical_candidate,True,retain_raw_code_and_review_semantic_encoding
8,9,IN_BYTES,quantitative_candidate,True,retain_quantitative_value
9,10,IN_PKTS,quantitative_candidate,True,retain_quantitative_value


In [14]:
# ---------------------------------------------------------------------
# Validate the provisional input policy
# ---------------------------------------------------------------------

input_policy_summary = pd.Series(
    {
        "total_source_columns": len(source_columns),
        "provisional_core_features": len(
            provisional_core_features
        ),
        "excluded_columns": len(
            excluded_core_columns
        ),
        "quantitative_core_features": sum(
            column
            in provisional_core_features
            for column in feature_roles.loc[
                feature_roles["feature_role"]
                == "quantitative_candidate",
                "column_name",
            ]
        ),
        "encoded_core_features": sum(
            column
            in provisional_core_features
            for column in encoded_categorical_columns
        ),
    },
    name="value",
)

assert len(source_columns) == 55
assert len(excluded_core_columns) == 7
assert len(provisional_core_features) == 48

assert not set(
    ground_truth_columns
).intersection(
    provisional_core_features
), (
    "Ground-truth leakage detected in the core feature list."
)

assert set(
    provisional_core_features
).isdisjoint(
    excluded_core_columns
), (
    "A column was assigned to both included and excluded sets."
)

assert set(
    provisional_core_features
).union(
    excluded_core_columns
) == set(source_columns), (
    "The inclusion and exclusion lists do not cover all source columns."
)

input_policy_summary

total_source_columns          55
provisional_core_features     48
excluded_columns               7
quantitative_core_features    37
encoded_core_features         11
Name: value, dtype: int64

## Duplicate and input-profile audit

Duplicate analysis is conducted at two levels.

### Exact source-record duplicates

Two records are exact source duplicates when all 55 source columns,
including identifiers, timestamps and ground truth, are equal.

### Duplicate model-input profiles

Two records have the same model-input profile when all 48 provisional
core-input fields are equal, even if their excluded identifiers,
timestamps or ground-truth fields differ.

A model-input profile appearing in both Benign and DoS records is a
cross-label conflict. Such records provide identical observable input
but different target answers under the provisional feature policy.

The following audit measures these patterns using deterministic 64-bit
row hashes. Hashing permits millions of records to be compared without
retaining the complete 55-column dataset in memory.

In [15]:
# ---------------------------------------------------------------------
# Scan Benign and DoS records for source and model-profile duplicates
# ---------------------------------------------------------------------

DUPLICATE_AUDIT_CHUNK_SIZE = 100_000

# All 55 fields define an exact source-record duplicate.
exact_source_comparison_columns = source_columns.copy()

# The 48 provisional input fields define what the model will actually see.
model_profile_columns = provisional_core_features.copy()

# Hashes are collected separately by category so that we can later detect
# profiles appearing in both Benign and DoS.
exact_hash_chunks = {
    category: []
    for category in TARGET_CATEGORIES
}

profile_hash_chunks = {
    category: []
    for category in TARGET_CATEGORIES
}

rows_processed = 0
target_rows_hashed = {
    category: 0
    for category in TARGET_CATEGORIES
}


for chunk_number, chunk in enumerate(
    pd.read_csv(
        SOURCE_CSV,
        chunksize=DUPLICATE_AUDIT_CHUNK_SIZE,
    ),
    start=1,
):
    # Only the provisional binary-task population is needed here.
    target_chunk = chunk[
        chunk["Attack"].isin(TARGET_CATEGORIES)
    ]

    for category in TARGET_CATEGORIES:
        category_chunk = target_chunk[
            target_chunk["Attack"] == category
        ]

        target_rows_hashed[category] += len(
            category_chunk
        )

        if category_chunk.empty:
            continue

        # Exact-source hash:
        # all 55 source fields participate in the comparison.
        exact_hash = pd.util.hash_pandas_object(
            category_chunk[
                exact_source_comparison_columns
            ],
            index=False,
        ).to_numpy(dtype=np.uint64)

        # Model-profile hash:
        # only the 48 fields that would be observable to the model
        # participate in the comparison.
        profile_hash = pd.util.hash_pandas_object(
            category_chunk[
                model_profile_columns
            ],
            index=False,
        ).to_numpy(dtype=np.uint64)

        exact_hash_chunks[category].append(
            exact_hash
        )

        profile_hash_chunks[category].append(
            profile_hash
        )

    rows_processed += len(chunk)

    if chunk_number == 1 or chunk_number % 5 == 0:
        print(
            f"Processed chunk {chunk_number:>2}: "
            f"{rows_processed:,} cumulative source rows"
        )


# Join the small per-chunk arrays into one array per category.
exact_hashes = {
    category: np.concatenate(
        exact_hash_chunks[category]
    )
    for category in TARGET_CATEGORIES
}

profile_hashes = {
    category: np.concatenate(
        profile_hash_chunks[category]
    )
    for category in TARGET_CATEGORIES
}


print("\nDuplicate audit scan completed.")
print(f"Total source rows scanned: {rows_processed:,}")

for category in TARGET_CATEGORIES:
    print(
        f"{category} records hashed: "
        f"{target_rows_hashed[category]:,}"
    )

Processed chunk  1: 100,000 cumulative source rows
Processed chunk  5: 500,000 cumulative source rows
Processed chunk 10: 1,000,000 cumulative source rows
Processed chunk 15: 1,500,000 cumulative source rows
Processed chunk 20: 2,000,000 cumulative source rows

Duplicate audit scan completed.
Total source rows scanned: 2,365,424
Benign records hashed: 2,237,731
DoS records hashed: 5,980


In [16]:
# ---------------------------------------------------------------------
# Summarise duplicate groups within each category
# ---------------------------------------------------------------------

duplicate_summary_rows = []

for category in TARGET_CATEGORIES:
    for comparison_level, hashes in [
        (
            "exact_source_record",
            exact_hashes[category],
        ),
        (
            "model_input_profile",
            profile_hashes[category],
        ),
    ]:
        unique_hashes, counts = np.unique(
            hashes,
            return_counts=True,
        )

        duplicate_group_counts = counts[
            counts > 1
        ]

        duplicate_summary_rows.append(
            {
                "Attack": category,
                "comparison_level": comparison_level,
                "total_records": len(hashes),
                "unique_patterns": len(unique_hashes),
                "duplicate_groups": len(
                    duplicate_group_counts
                ),
                # Includes the first occurrence and every repeated
                # occurrence belonging to a duplicate group.
                "records_in_duplicate_groups": int(
                    duplicate_group_counts.sum()
                ),
                # Number that would be removed if only one occurrence
                # of every duplicated pattern were retained.
                "duplicate_occurrences_beyond_first": int(
                    (duplicate_group_counts - 1).sum()
                ),
            }
        )

duplicate_audit_summary = pd.DataFrame(
    duplicate_summary_rows
)

duplicate_audit_summary

,Attack,comparison_level,total_records,unique_patterns,duplicate_groups,records_in_duplicate_groups,duplicate_occurrences_beyond_first
0,Benign,exact_source_record,2237731,2222930,14801,29602,14801
1,Benign,model_input_profile,2237731,2022399,90753,306085,215332
2,DoS,exact_source_record,5980,5971,9,18,9
3,DoS,model_input_profile,5980,5943,25,62,37


In [17]:
# ---------------------------------------------------------------------
# Detect model-input profiles shared by Benign and DoS
# ---------------------------------------------------------------------

benign_unique_profiles, benign_profile_counts = np.unique(
    profile_hashes["Benign"],
    return_counts=True,
)

dos_unique_profiles, dos_profile_counts = np.unique(
    profile_hashes["DoS"],
    return_counts=True,
)

# Return the positions of shared hashes in both unique-profile arrays.
(
    shared_profile_hashes,
    benign_shared_positions,
    dos_shared_positions,
) = np.intersect1d(
    benign_unique_profiles,
    dos_unique_profiles,
    assume_unique=True,
    return_indices=True,
)

cross_label_conflict_summary = pd.Series(
    {
        "shared_model_profiles": len(
            shared_profile_hashes
        ),
        "benign_records_in_shared_profiles": int(
            benign_profile_counts[
                benign_shared_positions
            ].sum()
        ),
        "dos_records_in_shared_profiles": int(
            dos_profile_counts[
                dos_shared_positions
            ].sum()
        ),
    },
    name="value",
)

cross_label_conflict_summary

shared_model_profiles                0
benign_records_in_shared_profiles    0
dos_records_in_shared_profiles       0
Name: value, dtype: int64

## Duplicate-audit findings and preparation decision

The duplicate audit distinguished exact source-record duplicates from
duplicate model-input profiles.

### Exact source-record duplicates

Among 2,237,731 Benign records, 14,801 duplicate occurrences beyond the
first copy were detected. Among 5,980 DoS records, nine duplicate
occurrences beyond the first copy were detected.

These are records for which all 55 source columns are equal.

### Duplicate model-input profiles

After excluding the seven non-model fields, the Benign population
contained 2,022,399 unique 48-field model-input profiles. Retaining one
representative of each profile would remove 215,332 additional Benign
occurrences.

The DoS population contained 5,943 unique model-input profiles.
Retaining one representative of each profile would remove 37 additional
DoS occurrences.

Duplicate model-input profiles are not necessarily erroneous source
records. They may represent distinct network flows that become
indistinguishable after identifiers and timestamps are excluded.

### Cross-label conflicts

No 48-field model-input profile was shared between the Benign and DoS
categories. Therefore, the duplicate audit found no instance in which
an identical provisional model input had conflicting binary ground-truth
labels.

### Development-sampling decision

The raw dataset will remain unchanged.

For the balanced development pilot, sampling will operate over unique
48-field model-input profiles rather than over all source-record
occurrences. One representative source row will be retained for each
selected profile.

This gives each distinct model input one sampling opportunity and avoids
repeatedly evaluating an identical prompt. This is a development-set
design decision and does not imply that repeated flows are invalid
observations.

For conventional-model evaluation, all records sharing a model-input
profile must remain in the same partition to prevent train/test leakage.

In [18]:
# ---------------------------------------------------------------------
# Export the provisional feature policy for use by later notebooks
# ---------------------------------------------------------------------

CONFIG_DIR = PROJECT_ROOT / "configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_POLICY_PATH = (
    CONFIG_DIR / "feature_policy_provisional.csv"
)

# This output contains only field names, roles and preparation decisions.
# It contains no network-flow records, IP addresses or ground-truth rows.
feature_roles.to_csv(
    FEATURE_POLICY_PATH,
    index=False,
)

print(
    "Provisional feature policy saved to:\n"
    f"{FEATURE_POLICY_PATH}"
)

print(
    "\nRows written: "
    f"{len(feature_roles)}"
)

Provisional feature policy saved to:
/Users/ruiwang/Developer/compsci742-rui-pilot/configs/feature_policy_provisional.csv

Rows written: 55


In [19]:
# ---------------------------------------------------------------------
# Verify that the exported feature policy can be read back correctly
# ---------------------------------------------------------------------

reloaded_feature_policy = pd.read_csv(
    FEATURE_POLICY_PATH
)

assert len(reloaded_feature_policy) == 55
assert reloaded_feature_policy["column_name"].is_unique
assert (
    reloaded_feature_policy[
        "provisional_core_input"
    ].sum()
    == 48
)

assert not reloaded_feature_policy.loc[
    reloaded_feature_policy[
        "column_name"
    ].isin(["Label", "Attack"]),
    "provisional_core_input",
].any(), (
    "Ground-truth fields were incorrectly included in model input."
)

policy_verification = pd.Series(
    {
        "policy_rows": len(
            reloaded_feature_policy
        ),
        "core_input_fields": int(
            reloaded_feature_policy[
                "provisional_core_input"
            ].sum()
        ),
        "excluded_fields": int(
            (
                ~reloaded_feature_policy[
                    "provisional_core_input"
                ]
            ).sum()
        ),
        "ground_truth_excluded": True,
        "policy_file_exists": (
            FEATURE_POLICY_PATH.exists()
        ),
    },
    name="value",
)

policy_verification

policy_rows                55
core_input_fields          48
excluded_fields             7
ground_truth_excluded    True
policy_file_exists       True
Name: value, dtype: object